In [0]:
dbutils.widgets.removeAll()

In [0]:
 %python
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "etl_vinkos")
dbutils.widgets.text("esquemabron", "bronze")
dbutils.widgets.text("storageName", "adlsvinkosetl")
dbutils.widgets.text("schemaSilver", "silver")
dbutils.widgets.text("esquemagold", "gold")



In [0]:
%python
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquemabron = dbutils.widgets.get("esquemabron")
storageName = dbutils.widgets.get("storageName")
schemaSilver = dbutils.widgets.get("schemaSilver")
esquemagold = dbutils.widgets.get("esquemagold")



ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/*.txt"

In [0]:

#Extraemos la tabla de Estadisticos y bitacora para colocarla en un df


from pyspark.sql.functions import col

# Tabla temporal con todos los registros a procesar
df_estadistica = spark.table(
    f"{catalogo}.{esquemabron}.estadistica_contenedor"
)

# Tabla de control de archivos ya procesados
df_bitacora = (
    spark.table(
        f"{catalogo}.{schemaSilver}.archivos_a_procesar"
    )
    #.select()
    .distinct()
)





display(df_estadistica)
display(df_bitacora)

In [0]:

#Revisamos los nom_archivos pendientes a cargar

from pyspark.sql.functions import col

df_procesar_estadisticas = (
    df_estadistica.alias("e")
    .join(
        df_bitacora.alias("b"),
        col("e.archivo_origen") == col("b.nom_archivo"),
        "inner"
    )
    .select("e.*")
)

display(df_procesar_estadisticas)

In [0]:
display(df_bitacora)

In [0]:
from pyspark.sql.functions import col

df_procesar_estadisticas = (
    df_estadistica.join(
        df_bitacora.select("nom_archivo"),
        df_estadistica["archivo_origen"] == df_bitacora["nom_archivo"],
        "inner"
    )
)


df_no_procesar_estadisticas = (
    df_estadistica.join(
        df_bitacora.select("nom_archivo"),
        df_estadistica["archivo_origen"] == df_bitacora["nom_archivo"],
        "left_anti"
    )
)

display(df_procesar_estadisticas)
display(df_no_procesar_estadisticas)


##Validaciones de DF "df_procesar_estadisticas"

In [0]:

from pyspark.sql.functions import *

'''
Validamos de la siguiente forma :

Validar Email.
Validar Fecha envío.
Validar Fecha open.
Validar Fecha click.

'''


# Expresión regular para Email
regex_email = r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$'

df_validado = (

    df_procesar_estadisticas

    #-------------------------------------------------------
    # EMAIL
    #-------------------------------------------------------
    .withColumn(
        "err_email",
        when(
            col("email").isNull() |
            (trim(col("email")) == "") |
            (~col("email").rlike(regex_email)),
            lit("EMAIL")
        )
    )

    #-------------------------------------------------------
    # FECHA ENVIO
    #-------------------------------------------------------
    .withColumn(
        "err_fe",
        when(
            col("fecha_envio").isNull() |
            (trim(col("fecha_envio")) == "") |
            (trim(col("fecha_envio")) == "-") |
            (
                to_timestamp(
                    col("fecha_envio"),
                    "dd/MM/yyyy HH:mm"
                ).isNull()
            ),
            lit("FE")
        )
    )

    #-------------------------------------------------------
    # FECHA OPEN
    #-------------------------------------------------------
    .withColumn(
        "err_fo",
        when(
            col("fecha_open").isNull() |
            (trim(col("fecha_open")) == "") |
            (trim(col("fecha_open")) == "-") |
            (
                to_timestamp(
                    col("fecha_open"),
                    "dd/MM/yyyy HH:mm"
                ).isNull()
            ),
            lit("FO")
        )
    )

    #-------------------------------------------------------
    # FECHA CLICK
    #-------------------------------------------------------
    .withColumn(
        "err_fc",
        when(
            col("fecha_click").isNull() |
            (trim(col("fecha_click")) == "") |
            (trim(col("fecha_click")) == "-") |
            (
                to_timestamp(
                    col("fecha_click"),
                    "dd/MM/yyyy HH:mm"
                ).isNull()
            ),
            lit("FC")
        )
    )

)

In [0]:

#Juntamos los errores en la columna error


df_validado = (
    df_validado
    .withColumn(
        "error",
        concat_ws(
            ",",
            col("err_email"),
            col("err_fe"),
            col("err_fo"),
            col("err_fc")
        )
    )
)

#Si no tiene error dejamos null ""

df_validado = (
    df_validado
    .withColumn(
        "error",
        when(
            col("error") == "",
            None
        ).otherwise(col("error"))
    )
)


#Eliminamos columnas auxiliares
df_validado = (
    df_validado
    .drop(
        "err_email",
        "err_fe",
        "err_fo",
        "err_fc"
    )
)

display(df_validado)

In [0]:
#Creamos el df de bitacora con los archivos procesados


df_estadistica_error = df_validado.filter(col("error").isNotNull())
df_estadistica_ok_ = df_validado.filter(col("error").isNull())

#Eliminamos columnas error
df_estadistica_ok = (
    df_estadistica_ok_
    .drop(
        "error"
    )
)



display(df_estadistica_error)
display(df_estadistica_ok)




###Se carga los df finales a las tablas productivas

In [0]:
#Se escribe en la bitacora los nuevos valores estadistica
df_estadistica_error.write \
    .mode("append") \
    .saveAsTable(f"{catalogo}.{esquemagold}.estadistica_log_error")

In [0]:

#Se hace el casteo del tipo de campo ya que al inicio solo se cargaron los campos de texto

from pyspark.sql.functions import col, to_timestamp

df_estadistica_ok_ = (
    df_estadistica_ok
    .select(
        col("email"),
        col("jyv"),
        col("badmail"),
        col("baja"),

        to_timestamp(col("fecha_envio"), "dd/MM/yyyy HH:mm").alias("fecha_envio"),
        to_timestamp(col("fecha_open"), "dd/MM/yyyy HH:mm").alias("fecha_open"),

        col("opens").cast("int").alias("opens"),
        col("opens_virales").cast("int").alias("opens_virales"),

        to_timestamp(col("fecha_click"), "dd/MM/yyyy HH:mm").alias("fecha_click"),

        col("clicks").cast("int").alias("clicks"),
        col("clicks_virales").cast("int").alias("clicks_virales"),

        col("links"),
        col("ips"),
        col("navegadores"),
        col("plataformas"),
        col("archivo_origen"),
        col("fecha_carga"),
        col("nom_archivo")
    )
)

display(df_estadistica_ok_)


In [0]:

#Se escribe en la bitacora los nuevos valores estadistica log de errores

df_estadistica_ok_.write \
    .mode("append") \
    .saveAsTable(f"{catalogo}.{esquemagold}.estadistica")

In [0]:
#Se escribe en la bitacora los nuevos valores bitacora_ctrl_file
df_bitacora.write \
    .mode("append") \
    .saveAsTable(f"{catalogo}.{esquemagold}.bitacora_ctrl_file")

###Truncamos tablas contenedor

In [0]:
%sql

TRUNCATE TABLE etl_vinkos.bronze.estadistica_contenedor;
TRUNCATE TABLE etl_vinkos.silver.archivos_a_procesar;
